In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier,DecisionTreeRegressor
from sklearn.linear_model import LogisticRegression
from tqdm import tqdm
from sklearn.ensemble import VotingClassifier,VotingRegressor
from sklearn.metrics import log_loss,accuracy_score,precision_score,f1_score,mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder,LabelEncoder,StandardScaler,MinMaxScaler
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer,make_column_selector


In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
hr=pd.read_csv("D:\Machine Learning\Datasets\HR_comma_sep.csv")

In [4]:

X , y = hr.drop('left',axis=1) , hr['left']
ohe = OneHotEncoder(drop='first', sparse_output=False).set_output(transform="pandas")
col_transformer = ColumnTransformer([("OHE", ohe, make_column_selector(dtype_include=object))], 
                                    remainder='passthrough', 
                                    verbose_feature_names_out=False)
col_transformer = col_transformer.set_output(transform="pandas")
X = col_transformer.fit_transform(X)


In [5]:
X_train , X_test , y_train , y_test = train_test_split(X,y,test_size=0.3,random_state=25,stratify=y)

In [6]:
dtc1=DecisionTreeClassifier(random_state=25,max_depth=3)
dtc2=DecisionTreeClassifier(random_state=25,max_depth=2)
lr=LogisticRegression()
voting=VotingClassifier(estimators=[('TREE1',dtc1),('TREE2',dtc2),('LR',lr)])
voting.fit(X_train,y_train)
y_pred=voting.predict(X_test)
print(accuracy_score(y_test,y_pred))


0.9017559457657257


# Giving Average of three models summary

In [7]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# scaler to scale data
std_scaler=StandardScaler()

#knn model
knn1= KNeighborsClassifier(n_neighbors=5)

#pipe1->first scale and data to knn
pipe1=Pipeline([('SCL',std_scaler),('KNN',knn1)])
pipe1.fit(X_train,y_train)

#svc model
svm=SVC(kernel="rbf")

#pipe2->first scale and data to svm
pipe2=Pipeline([('SCL',std_scaler),('SVM',svm)])
pipe2.fit(X_train,y_train)

dtc=DecisionTreeClassifier(random_state=25,max_depth=4)

#pipe3->first scale and data to knn
pipe3=Pipeline([('SCL',std_scaler),('DTC',dtc)])
pipe3.fit(X_train,y_train)

#voting is also like pipeline 
voting=VotingClassifier([('DTC',dtc),('SVM',svm),('KNN',knn1)])
voting.fit(X_train,y_train)

y_pred=voting.predict(X_test)
# avaerage of these three models accuracy_score
print('Accuracy score: ',accuracy_score(y_test,y_pred))

Accuracy score:  0.9773282951767059


In [8]:
std_scaler=StandardScaler()

knn1= KNeighborsClassifier(n_neighbors=5)
pipe1=Pipeline([('SCL',std_scaler),('KNN',knn1)])
pipe1.fit(X_train,y_train)

svm=SVC(kernel="rbf",probability=True,random_state=25)
pipe2=Pipeline([('SCL',std_scaler),('SVM',svm)])
pipe2.fit(X_train,y_train)

dtc=DecisionTreeClassifier(random_state=25,max_depth=4)
pipe2=Pipeline([('SCL',std_scaler),('DTC',dtc)])
pipe2.fit(X_train,y_train)

voting=VotingClassifier([('DTC',dtc),('SVM',svm),('KNN',knn1)],voting='soft')
voting.fit(X_train,y_train)
y_pred=voting.predict(X_test)

print('Accuracy score: ',accuracy_score(y_test,y_pred))

Accuracy score:  0.9715492331629251


In [9]:
sc = StandardScaler()
knn1 = KNeighborsClassifier(n_neighbors=5)
knn2 = KNeighborsClassifier(n_neighbors=3)
lr = LogisticRegression(max_iter=5000)
dct1 = DecisionTreeClassifier(random_state=25, max_depth=4)
dct2 = DecisionTreeClassifier(random_state=25, max_depth=7)
pipeknn1 = Pipeline([("scaler", sc), ("knn", knn1)])
pipeknn2 = Pipeline([("scaler", sc), ("knn", knn2)])
voting = VotingClassifier(estimators=[("KNN1", pipeknn1), ("KNN2", pipeknn2), ("LR", lr), ("DCT1", dct1), ("DCT2", dct2)], voting='soft')
voting.fit(X_train, y_train)
y_pred_proba = voting.predict_proba(X_test)
logLoss = log_loss(y_test, y_pred_proba)
print(f"Voting Classifier with Multiple KNN and DCT Log Loss: {logLoss:.4f}")

Voting Classifier with Multiple KNN and DCT Log Loss: 0.1250


# HyperParameter Tuning

In [10]:
depths = [2, 4, 6, 8, 10, 12, 14, 16, 18, 20]
neighbors = [1, 3, 5, 7, 9, 11, 13, 15, 17, 19]
scores = []
for depth in tqdm(depths):
    for neighbor in neighbors:
        dct = DecisionTreeClassifier(random_state=25, max_depth=depth)
        knn = KNeighborsClassifier(n_neighbors=neighbor)
        lr = LogisticRegression(max_iter=5000)
        pipeknn = Pipeline([("scaler", std_scaler), ("knn", knn)])
        voting = VotingClassifier(estimators=[("KNN", pipeknn), ("DCT", dct), ("LR", lr)], voting='soft')
        voting.fit(X_train, y_train)
        y_pred = voting.predict(X_test)
        score = log_loss(y_test, y_pred)
        scores.append([depth, neighbor, score])
df_scores = pd.DataFrame(scores, columns=["Max_Depth", "N_Neighbors", "score"])
df_scores.sort_values("score")

100%|██████████| 10/10 [00:43<00:00,  4.33s/it]


,Max_Depth,N_Neighbors,score
80,18,1,0.640918
90,20,1,0.648930
70,16,1,0.672964
60,14,1,0.705010
81,18,3,0.729045
...,...,...,...
5,2,11,2.123042
6,2,13,2.251226
7,2,15,2.339352
8,2,17,2.395433


# Boston Dataset

In [11]:
boston=pd.read_csv("D:\Machine Learning\Datasets\Boston.csv")
X=boston.drop('medv',axis=1)
y=boston['medv']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [12]:
from sklearn.linear_model import Lasso,Ridge,ElasticNet



In [13]:
# MSE of Elasticnet
ElasticNet=ElasticNet()
ElasticNet.fit(X_train,y_train)
y_pred=ElasticNet.predict(X_test)
print(mean_absolute_error(y_test,y_pred))

3.2440006415369758


In [14]:
# MSE of Ridge
ridge=Ridge()
ridge.fit(X_train,y_train)
y_pred=ridge.predict(X_test)
print(mean_absolute_error(y_test,y_pred))


3.1329474278055365


In [15]:
# MSE of Lasso
Lasso=Lasso()
Lasso.fit(X_train,y_train)
y_pred=Lasso.predict(X_test)
print(mean_absolute_error(y_test,y_pred))

3.2535397673681623


In [16]:
## MSE of DecisionTreeRegressor
dtr=DecisionTreeRegressor(random_state=25)
dtr.fit(X_train,y_train)
y_pred=dtr.predict(X_test)
print(mean_absolute_error(y_test,y_pred))

2.56078431372549


In [22]:

# Scale the data
from sklearn.metrics import mean_squared_error

from sklearn.linear_model import ElasticNet
sc = StandardScaler()
X_train_scaled = sc.fit_transform(X_train)
X_test_scaled = sc.transform(X_test)

# Initialize models
ridge = Ridge()
elastic = ElasticNet()
dtr = DecisionTreeRegressor(random_state=25)

# Create voting regressor
voting = VotingRegressor(estimators=[("RIDGE", ridge), ("ELASTIC", elastic), ("TREE", dtr)])
voting.fit(X_train_scaled, y_train)

# Predict and evaluate
y_pred = voting.predict(X_test_scaled)

# Calculate both metrics
print(f"MAE: {mean_absolute_error(y_test, y_pred):.4f}")
print(f"MSE: {mean_squared_error(y_test, y_pred):.4f}")


MAE: 2.6151
MSE: 17.1613


In [24]:
# weights acts preference , the model perform best give more weight
sc=StandardScaler()

ridge=Ridge()

elastic=ElasticNet()

dtr=DecisionTreeRegressor(random_state=25)

voting = VotingRegressor(estimators=[("RIDGE",ridge),("ELASTIC",elastic),("TREE",dtr)],weights=[5,2,7])
voting.fit(X_train,y_train)
y_pred=voting.predict(X_test)
# Calculate both metrics
print(f"MAE: {mean_absolute_error(y_test, y_pred):.4f}")
print(f"MSE: {mean_squared_error(y_test, y_pred):.4f}")


MAE: 2.3846
MSE: 15.3111
